# Vision-Transformer CRP — Experiments

Companion to [`walkthrough.ipynb`](walkthrough.ipynb). Drives the milestone sweep scripts in [`experiments/`](../../experiments) and visualises their results, all in one place.

## What you'll see

1. **Setup + dataset selection** — same `DATASET_NAME` knob as the walkthrough.
2. **Run sweeps** — call the milestone drivers from Python (or skip this and use cached CSVs from previous runs).
3. **Tables** — the per-rule pass/fail matrix the milestone drivers print to stdout, this time as a pandas DataFrame you can filter and slice.
4. **Plots** — del_gap / ins_gap bar charts per granularity, with the random-baseline gap shown for each rule.

Each milestone's findings narrative lives in [`CURRENT_STATE.md`](../../CURRENT_STATE.md); this notebook is the visual companion.

## 1. Setup

Imports and path bootstrap. Same convention as the walkthrough — `<repo>/data/` for artefacts, `experiments/` on `sys.path` so we can `import` the milestone drivers as Python modules.

In [ ]:
from __future__ import annotations

import csv
import sys
from collections import defaultdict
from pathlib import Path
from statistics import mean

import matplotlib.pyplot as plt
import numpy as np

def _repo_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / 'pyproject.toml').is_file():
            return p
        p = p.parent
    raise RuntimeError('repo root not found')

REPO_ROOT = _repo_root()
DATA_DIR = REPO_ROOT / 'data'
EXP_DIR = REPO_ROOT / 'experiments'
sys.path.insert(0, str(EXP_DIR))

print(f'repo: {REPO_ROOT}')
print(f'data: {DATA_DIR}')

## 2. Configuration

Choose the dataset and the sweeps to visualise. The defaults match the milestone-A/D/G CSV layout shipped under `data/`.

* `DATASET_NAME` — `'imagenette'` for fast iteration (~10 min for a Milestone-A sweep), `'imagenet_val'` for the final benchmark (~1 hour per ε sweep, gated; see `experiments/datasets.py`).
* `MILESTONE_CSVS` — paths under `data/`. Missing files are simply skipped (with a warning); run the corresponding milestone driver to populate them.

In [ ]:
DATASET_NAME = 'imagenette'           # 'imagenette' | 'imagenet_val'

MILESTONE_CSVS = {
    'A':  DATA_DIR / 'milestone_a_results.csv',
    'D':  DATA_DIR / 'milestone_d_results.csv',
    'G':  DATA_DIR / 'milestone_g_results.csv',
}

for tag, path in MILESTONE_CSVS.items():
    print(f'milestone {tag}: {"" if path.exists() else "(missing) "}{path}')

## 3. Run a sweep (optional)

Skip this section to just visualise pre-cached CSVs. To re-run a milestone end-to-end inside the notebook, uncomment the block below — each driver exposes a `main()` callable but also runs as a subprocess via `python -m`. For long sweeps prefer the CLI from a terminal so per-image progress lines aren't buffered into the notebook.

Reference invocations:

```bash
uv run python experiments/run_milestone_a.py --dataset {dataset}
uv run python experiments/run_milestone_d.py --models vit_small_patch16_224 --dataset {dataset}
uv run python experiments/run_milestone_g.py --models vit_small_patch16_224 --dataset {dataset}
```

In [ ]:
# Uncomment to run Milestone A from inside the notebook (slow!).
# import subprocess
# subprocess.run([
#     'uv', 'run', 'python', str(EXP_DIR / 'run_milestone_a.py'),
#     '--dataset', DATASET_NAME,
#     '--model', 'vit_tiny_patch16_224',  # tiny for a quick smoke
#     '--n-per-class', '2', '--steps', '4',
# ], check=True, cwd=REPO_ROOT)

## 4. Per-rule verdict table

Reduce each CSV to one row per `(model, granularity, rule)` triple, compute mean del/ins AUC for `mode=true` and `mode=random`, and report the gap and verdict (the same matrix the drivers print to stdout). Rows are pivoted by rule for easy comparison.

In [ ]:
GRANULARITIES = ('head', 'head_dim', 'kqv_head', 'kqv_head_dim')

def _read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))

def _aggregate(rows):
    """Return [{'model','granularity','rule','del_true','del_rand','ins_true',
    'ins_rand','del_gap','ins_gap','verdict','n_images'}, ...]"""
    # Build a (rule_label) per row.
    out = []
    grouped = defaultdict(list)
    for r in rows:
        rule = r.get('residual_lrp') or r.get('palrp') or '-'
        if rule in ('False', 'True'): rule = f'palrp={rule}'
        if r.get('gamma') not in (None, '', 'None'):
            rule = f"γ={r['gamma']}"
        elif r.get('composite') == 'AttnLRPEpsilonComposite':
            rule = rule if rule != '-' else 'ε-LRP'
        model = r.get('model', '?')
        key = (model, r['concept_def'], rule)
        grouped[key].append(r)
    for (model, cd, rule), grp in grouped.items():
        true_rows = [r for r in grp if r['mode'] == 'true']
        rand_rows = [r for r in grp if r['mode'] == 'random']
        if not true_rows or not rand_rows: continue
        d_t = mean(float(r['deletion_auc'])  for r in true_rows)
        d_r = mean(float(r['deletion_auc'])  for r in rand_rows)
        i_t = mean(float(r['insertion_auc']) for r in true_rows)
        i_r = mean(float(r['insertion_auc']) for r in rand_rows)
        verdict = ('OK' if d_t < d_r and i_t > i_r
                   else ('del_FAIL' if d_t >= d_r else 'ins_FAIL'))
        out.append(dict(model=model, granularity=cd, rule=rule,
                        del_true=d_t, del_rand=d_r, del_gap=d_r - d_t,
                        ins_true=i_t, ins_rand=i_r, ins_gap=i_t - i_r,
                        verdict=verdict, n_images=len(true_rows)))
    return out

ALL_AGG = []
for tag, path in MILESTONE_CSVS.items():
    if not path.exists():
        print(f'milestone {tag}: skipping (no CSV at {path})')
        continue
    rows = _read_csv(path)
    agg = _aggregate(rows)
    print(f'milestone {tag}: {len(rows):4d} CSV rows → {len(agg):3d} (model, gran, rule) cells')
    for a in agg: a['milestone'] = tag
    ALL_AGG.extend(agg)

In [ ]:
# Compact verdict table — one block per (milestone, model).
if not ALL_AGG:
    print('No CSVs found. Run a milestone driver first; see Section 3.')
else:
    by_block = defaultdict(list)
    for a in ALL_AGG:
        by_block[(a['milestone'], a['model'])].append(a)

    for (tag, model), aggs in sorted(by_block.items()):
        rules = sorted({a['rule'] for a in aggs})
        rule_grans = {(a['rule'], a['granularity']): a for a in aggs}
        print(f'\n=== Milestone {tag} | {model} ===')
        header = f'{"granularity":>14}  '
        for rl in rules:
            header += f'{rl:>14}  '
        print(header)
        for g in GRANULARITIES:
            row = f'{g:>14}  '
            for rl in rules:
                a = rule_grans.get((rl, g))
                if not a:
                    row += f'{" — ":>14}  '
                else:
                    row += f'{a["verdict"]:>5} {a["del_gap"]:+.4f} {a["ins_gap"]:+.4f}  '
            print(row)
        print(f'(n={aggs[0]["n_images"]} per cell; cell shows del_gap (rand-true) ',
              'and ins_gap (true-rand) — both should be positive for OK)')

## 5. del_gap / ins_gap bar charts

For each milestone × model, plot the `(rand − true)` deletion gap and `(true − rand)` insertion gap per granularity, grouped by rule. Bars above zero satisfy the milestone-A acceptance criterion; bars below zero indicate the rule-granularity cell where random wins.

In [ ]:
if ALL_AGG:
    by_block = defaultdict(list)
    for a in ALL_AGG:
        by_block[(a['milestone'], a['model'])].append(a)

    for (tag, model), aggs in sorted(by_block.items()):
        rules = sorted({a['rule'] for a in aggs})
        if len(rules) == 0: continue
        x = np.arange(len(GRANULARITIES))
        width = max(0.12, 0.8 / max(len(rules), 1))
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
        for r_i, rl in enumerate(rules):
            del_vals = [next((a['del_gap'] for a in aggs if a['rule'] == rl and a['granularity'] == g), 0.0) for g in GRANULARITIES]
            ins_vals = [next((a['ins_gap'] for a in aggs if a['rule'] == rl and a['granularity'] == g), 0.0) for g in GRANULARITIES]
            axes[0].bar(x + r_i * width, del_vals, width, label=rl)
            axes[1].bar(x + r_i * width, ins_vals, width, label=rl)
        for ax, title in zip(axes, ('del_gap = rand − true', 'ins_gap = true − rand')):
            ax.set_xticks(x + width * (len(rules) - 1) / 2)
            ax.set_xticklabels(GRANULARITIES, rotation=30, ha='right')
            ax.axhline(0.0, color='k', lw=0.6)
            ax.set_title(title, fontsize=10)
            ax.legend(fontsize=8, loc='best')
        fig.suptitle(f'Milestone {tag}  •  {model}', fontsize=11)
        plt.tight_layout(); plt.show()

## 6. Notes

* The acceptance criterion (Milestone A) is `del_gap > 0` AND `ins_gap > 0` for all four granularities under one rule. As of iter-9, the only rule that hits this for `vit_small` is `residual_lrp='ratio'`; see `CURRENT_STATE.md` § "Milestone G".
* For `imagenet_val` runs, expect bigger error bars per cell (1 image per class instead of 16) but better cross-class diversity. Re-run with `--n-per-class 2` for tighter intervals at ~2× cost.
* The CSVs are pivotable on `(model, image, granularity, composite, residual_lrp, palrp, mode)` — use `pandas.read_csv` if you want richer slicing than the simple aggregator above.